<h3>Analysis of Insight Stimulating Examples</h3>

This notebook provides scripts used to anaylse a sample question taken from Software Recommendations Stack Exchange site and answers extracted from SORD (Comments_Contain). <br>
<p style="color: red;"> We are assuming here that the readers have downloaded and preprocessed the Software Recommendations data dump published in October 2025. As each updated version of data dump published on any Stack Exchange site, overrides the previous version, the October 2025 data dump may not be directly available for download on the official Software Recommendations Stack Exchange site. However, the data dump was made available by the authors as part of their replication package accompanying the following paper. The same replication package contains notebooks to preprocess the data dump. The procedure is similar to the one we used in this paper for preprocessing the Stack Overflow data dump except the fact that the Software Recommendations data dump is way too smaller than Stack Overflow and hence splitting the files to smaller chunks is not required.  </p><br>
<i>Fatima, Arjumand, and Onaiza Maqbool. "On the Sustainability of Software Recommendations: Analyzing the Least-Answered Site on the Stack Exchange Network." Data 11.3 (2026): 58.</i>

<h4>1. Getting the top 5 development related tags on Software Recommendations</h4>

We used the following SQL query to get the top 20 tags on Software Recommendations and manually identified the top 5 development related tags.

```sql
SELECT TOP (20) [AutoIdPK]
      ,[Id]
      ,[TagName]
      ,[TagCount]
      ,[ExcerptPostId]
      ,[WikiPostId]
  FROM [SoftwareRecommendationsDb_Oct2025].[dbo].[SoftwareRecommendationsTags]
  order by TagCount desc
```

<h4>2. Extracting top 5 software development related questions asked on Software Recommendations</h4>

```sql
SELECT TOP (5) [Id]
			,[CreationDate]
			,[Score]
			,[ViewCount]
			,[Title]
			,[Tags]
			,[AnswerCount]
			,[CommentCount]
			,[AcceptedAnswerId]
  FROM [SoftwareRecommendationsDb_Oct2025].[dbo].[SR_Qs]
  where (Tags like '%web-apps%' or   Tags like '%library%' or   Tags like '%python%' or   Tags like '%javascript%' or   Tags like '%java%' )
  and AcceptedAnswerId !=0
  order by viewcount desc, score desc, AnswerCount desc
```

<h4>3. Extracting sample software development question asked on Software Recommendations site</h4>

Out of the top 5 questions we considered two questions for further analysis with Id 32612 and 7463. Although the one with Id=32612 had higher score than the other one and had equal number of answers, the second one had greater commulative answer score. Hence, we selected the second question (wtih Id=7463) for further analysis. The following SQL query was used to calculate commulative answer score for the top 5 questions identified above.

```sql 
SELECT 
    q.id AS q_id,
    COALESCE(SUM(a.score), 0) AS total_a_score
FROM [SoftwareRecommendationsDb_Oct2025].[dbo].[SoftwareRecommendationsPosts] q
LEFT JOIN [SoftwareRecommendationsDb_Oct2025].[dbo].[SoftwareRecommendationsPosts] a 
    ON a.parentid = q.id
WHERE q.parentid = 0 and q.id in (32612, 13856, 14797, 7463, 17812)
GROUP BY q.id;
```

<h4>4. Extracting Relevant Comments from the Orginial Stack Overflow Data Dump's Comments Table</h4>

We found 1068 comments from the Comments table using the following SQL query.

```sql
SELECT  [AutoIdPK]
      ,[Id]
      ,[PostId]
      ,[Score]
      ,[Text]
      ,[CreationDate]
      ,[UserDisplayName]
      ,[UserId]
      ,[ContentLicense]
  FROM [StackOverflowCommentsDb_Oct2025].[dbo].[StackOverflowComments]
       where text like '%python%' and 
             text like '%read%' and
             text like '%csv%' and
             text like '%library%'
```

<h4>5. Extracting Recommendation Related Comments from SORD</h4>

We found 60 comments from Comments_Contain table using the following SQL query.

```sql
SELECT  [AutoIdPK]
      ,[Id]
      ,[PostId]
      ,[Score]
      ,[Text]
      ,[CreationDate]
      ,[UserDisplayName]
      ,[UserId]
      ,[ContentLicense]
  FROM [SORecommendationsDb].[dbo].[Comments_Contain]
      where text like '%python%' and 
            text like '%read%' and
            text like '%csv%' and
            text like '%library%'
      order by CreationDate
```      

Similarly, we found 24 comments from Comments_LikeMinusContain table using the following SQL query.

```sql
SELECT  [AutoIdPK]
      ,[Id]
      ,[PostId]
      ,[Score]
      ,[Text]
      ,[CreationDate]
      ,[UserDisplayName]
      ,[UserId]
      ,[ContentLicense]
  FROM [SORecommendationsDb].[dbo].[Comments_LikeMinusContain]
      where text like '%python%' and 
            text like '%read%' and
            text like '%csv%' and
            text like '%library%'
      order by CreationDate
```      

<h4>6. Finding the Mentions of Software Recommendation Question in Stack Overflow Data Dump Comments</h4>

We found that the original question asked on software recommendations was mentioned 9 times in Stack Oveflow data dump Comments.

```sql
SELECT  [AutoIdPK]
      ,[Id]
      ,[PostId]
      ,[Score]
      ,[Text]
      ,[CreationDate]
      ,[UserDisplayName]
      ,[UserId]
      ,[ContentLicense]
  FROM [StackOverflowCommentsDb_Oct2025].[dbo].[StackOverflowComments]
  where text like '%softwarerecs.stackexchange.com/questions/7463/%'
```

<h4>7. Finding the Mentions of Software Recommendation Question in SORD Comments_LikeMinusContain</h4>

Out of 24 relevant comments filtered from Comments_LikeMinusContain, 2 comments referred to the original question asked on Software Recommendations site. We used the following query to get these 2 comments. 

```sql
SELECT TOP (1000) [AutoIdPK]
      ,[Id]
      ,[PostId]
      ,[Score]
      ,[Text]
      ,[CreationDate]
      ,[UserDisplayName]
      ,[UserId]
      ,[ContentLicense]
  FROM [SORecommendationsDb].[dbo].[Comments_LikeMinusContain]
   where text like '%softwarerecs.stackexchange.com/questions/7463/%'
```

<h4>8. Finding the distribution of each of the 19 recommendation related keywords in SORD</h4>

```sql
SELECT  
    k.keyword,  

    (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[QuestionsTitle_Contain] t1 
     WHERE LOWER(t1.Title ) LIKE '%' + LOWER(k.keyword) + '%') AS QuestionsTitle_Contain,

    (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[QuestionsTitle_LikeMinusContain] t2 
     WHERE LOWER(t2.Title) LIKE '%' + LOWER(k.keyword) + '%') AS QuestionsTitle_LikeMinusContain,

    (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[QuestionsBody_Contain] t3 
     WHERE LOWER(t3.Body) LIKE '%' + LOWER(k.keyword) + '%') AS QuestionsBody_Contain,

	 (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[QuestionsBody_LikeMinusContain] t4 
     WHERE LOWER(t4.Body) LIKE '%' + LOWER(k.keyword) + '%') AS QuestionsBody_LikeMinusContain,

	 (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[Answers_Contain] t5 
     WHERE LOWER(t5.Body) LIKE '%' + LOWER(k.keyword) + '%') AS Answers_Contain,

	 (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[Answers_LikeMinusContain] t6 
     WHERE LOWER(t6.Body) LIKE '%' + LOWER(k.keyword) + '%') AS Answers_LikeMinusContain,

	 (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[Comments_Contain] t7 
     WHERE LOWER(t7.Text) LIKE '%' + LOWER(k.keyword) + '%') AS Comments_Contain,

	 (SELECT COUNT(*) 
     FROM [SORecommendationsDb].[dbo].[Comments_LikeMinusContain] t8 
     WHERE LOWER(t8.Text) LIKE '%' + LOWER(k.keyword) + '%') AS Comments_LikeMinusContain

FROM (VALUES  
    ('recommend'),  
    ('advise'),
	('advocate'), 
	('adopt'), 
	('suggest'), 
	('suitable'), 
	('praise'), 
	('favor'), 
	('support'), 
	('urge'), 
	('promote'), 
	('champion'), 
	('endorse'), 
	('commend'), 
	('propose'), 
	('oppose'), 
	('condemn'), 
    ('reject'),
	('disapprove')
) AS k(keyword);
```